# SSL preprocessing and unlabeled → labeled (nnU-Net LabelRun)

**Research software — not for clinical use.** Branch `ssl_sep`.

This notebook shows the **locked intensity recipe** and how **1980 unlabeled** CTs were turned into **nnU-Net pseudo-labels**. It does **not** produce the 0.4965 locked-test number.

| Campaign | What happened | Claim metric |
|---|---|---|
| **Labeling (this notebook)** | Teacher `nnunet_fold0` (locked-test **0.454723**) inferred masks on unlabeled volumes. Pool **1980** (562 anybleed / 1418 nobleed). Planned **1000**, finished **800**. | Pseudo-labels only |
| **Training (see `docs/ssl/README.md`)** | Supervised nnU-Net 5-fold on **163** locked train+val labeled cases. Compute limited us to **2 folds**. | fold0 ~ep510 **0.4802**; fold1 ep511 **0.4965** (best); fold1 ep679 **0.4418** |

Locked test (n=29) never received unlabeled IDs and was never used for fold training.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "preprocessing.yaml").is_file():
    ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else ROOT.parent
if not (ROOT / "config" / "preprocessing.yaml").is_file():
    ROOT = Path(r"C:\Users\1016f\OneDrive\Desktop\BrainHemorrhageAI-ssl_sep")

src = ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from preprocessing.preprocess_volume import (
    apply_intensity_clip,
    apply_normalization,
    load_preprocessing_config,
    preprocess_volume,
    resolve_clip_bounds,
)

print("project_root:", ROOT)
assert (ROOT / "config" / "preprocessing.yaml").is_file()

## Locked preprocessing (do not change)

Authority: `config/preprocessing.yaml` and `src/preprocessing/preprocess_volume.py`.

- HU clip **[-40, 120]**
- Normalize **(HU − 40) / 80** (`normalization_method: zscore`, `train_mean: 40`, `train_std: 80`)
- **No resampling** — native in-plane 512×512, original spacing / affine kept

nnU-Net training still runs its own `nnUNetv2_plan_and_preprocess` internally. Locked-test scoring always compares masks on the **native GT grid**.

In [ ]:
config = load_preprocessing_config(ROOT / "config" / "preprocessing.yaml")
clip_min, clip_max = resolve_clip_bounds(config)
print("clip:", clip_min, clip_max)
print("normalization_method:", config["normalization_method"])
print("train_mean / train_std:", config["train_mean"], config["train_std"])
assert (clip_min, clip_max) == (-40.0, 120.0)
assert float(config["train_mean"]) == 40.0 and float(config["train_std"]) == 80.0

# Synthetic HU slice: prove clip + (HU-40)/80 without requiring volumes on disk.
hu = np.array([-1000.0, -40.0, 40.0, 120.0, 400.0], dtype=np.float32)
clipped = apply_intensity_clip(hu, clip_min, clip_max)
normalized = apply_normalization(clipped, config)
print("HU       ", hu.tolist())
print("clipped  ", clipped.tolist())
print("normalized", [round(float(v), 4) for v in normalized])
assert clipped.min() >= -40 and clipped.max() <= 120
assert np.isclose(normalized[2], 0.0)  # HU 40 → 0
assert np.isclose(normalized[1], -1.0)  # HU -40 → -1
assert np.isclose(normalized[3], 1.0)  # HU 120 → 1

In [ ]:
images_dir = ROOT / "data" / "raw" / "label_192" / "images"
splits = pd.read_csv(ROOT / "data" / "metadata" / "splits.csv")
print("locked split counts:", dict(Counter(splits["split"])))
assert dict(Counter(splits["split"])) == {"train": 134, "val": 29, "test": 29}

if images_dir.is_dir():
    example = splits.loc[splits["split"] == "train", "filename"].iloc[0]
    result = preprocess_volume(example, config_path=ROOT / "config" / "preprocessing.yaml")
    print("example:", result.filename)
    print("native shape:", result.original_shape, "spacing:", result.original_spacing)
    print(
        "processed image:",
        result.processed_image.dtype,
        "min", float(result.processed_image.min()),
        "max", float(result.processed_image.max()),
    )
    print("mask labels:", sorted(int(v) for v in np.unique(result.processed_mask)))
    assert result.original_shape[:2] == (512, 512)
else:
    print("label_192 not mounted under data/raw — skip live volume demo.")
    print("Place BHSD images at", images_dir)

## Unlabeled pool → nnU-Net pseudo-labels

The unlabeled set is **1980** volumes with **zero** overlap against locked train/val/test.

Teacher LabelRun (`nnunet_fold0`):

- Planned **1000** (562 anybleed + 438 nobleed; 6 Phase-7 failures excluded)
- Finished **800 ok / 0 err** (stopped before the remaining ~200)
- T4 wall time **~52 s/case** → **~11–12 h** for 800 cases (12 h Kaggle session cap)

Those 800 masks are **pseudo-labels**. They are not the 5-fold training set and they are not the 0.4965 locked-test claim.

In [ ]:
summary = json.loads(
    (ROOT / "data" / "metadata" / "unlabeled_inventory_summary.json").read_text(encoding="utf-8")
)
counts = summary["counts"]
print("unlabeled volumes:", counts["unlabeled_volumes"])
print("subgroups:", counts["subgroups"])
print("labeled 192 split:", summary["leakage"]["labeled_split_counts"])
print("leakage-free:", summary["leakage"]["labeled_leakage_free"])
print("LabelRun:", summary["label_run"])

assert counts["unlabeled_volumes"] == 1980
assert counts["subgroups"] == {"anybleed": 562, "nobleed": 1418}
assert summary["leakage"]["labeled_leakage_free"] is True
assert summary["label_run"]["finished_ok"] == 800
assert summary["label_run"]["planned"] == 1000
assert summary["label_run"]["phase7_failures_excluded"] == 6
assert summary["label_run"]["leakage_vs_locked_splits"] == 0

## 5-fold labeled split (training campaign, not this LabelRun)

The Dice numbers in `docs/ssl/README.md` come from **supervised** nnU-Net training on labeled `label_192` only:

- **163** = locked train+val (test never included)
- fingerprint `4714804f9f1cabd9`
- Different from V1 `nnunet_splits_final.json` (134/29 matched to MONAI)

Limited compute stopped us at **2 folds** of a planned 5.

In [ ]:
meta = json.loads(
    (ROOT / "data" / "metadata" / "nnunet_splits_5fold_meta.json").read_text(encoding="utf-8")
)
folds = json.loads(
    (ROOT / "data" / "metadata" / "nnunet_splits_5fold.json").read_text(encoding="utf-8")
)
test_ids = {
    Path(name).stem.replace(".nii", "")
    for name in splits.loc[splits["split"] == "test", "filename"]
}
# splits.csv uses full filenames; 5-fold JSON uses case ids without .nii.gz
test_case_ids = {str(n).replace(".nii.gz", "") for n in splits.loc[splits["split"] == "test", "filename"]}

print("n_dev:", meta["n_dev"], "fingerprint:", meta["fingerprint"])
assert meta["n_dev"] == 163
assert meta["fingerprint"] == "4714804f9f1cabd9"
assert len(folds) == 5

for i, fold in enumerate(folds):
    train_n, val_n = len(fold["train"]), len(fold["val"])
    overlap = set(fold["train"]) & set(fold["val"])
    test_hit = (set(fold["train"]) | set(fold["val"])) & test_case_ids
    print(f"fold {i}: train={train_n} val={val_n} train∩val={len(overlap)} test_leak={len(test_hit)}")
    assert not overlap
    assert not test_hit
    assert train_n + val_n == 163

print("locked test never included in 5-fold CV.")

## What this notebook does **not** claim

- LabelRun 800 ≠ the 0.4965 model. That score is **supervised fold1 epoch 511** on the locked 29-case test set.
- Phase 8 MONAI student distillation (Control A vs Treatment B) was **not run**.
- The UI model `nnunet_ssl_2fold` **runs** a softmax ensemble of fold0 + fold1. A pooled **ensemble** locked-test Dice has **not** been scored; do not invent one.

Next: [`docs/ssl/README.md`](../docs/ssl/README.md).